# Catboost Optuna

In [1]:
import polars as pl
import pandas as pd
import numpy as np
import optuna
from catboost import CatBoostRegressor, Pool
from sklearn.metrics import mean_squared_error
from pathlib import Path

FEATURES_DIR = Path("../data/processed/features_v5")

def rmsle_score(y_true, y_pred):
    return np.sqrt(mean_squared_error(np.log1p(np.clip(y_true, 0, None)),
                                      np.log1p(np.clip(y_pred, 0, None))))

def load_fold(fold_path: Path) -> pd.DataFrame:
    return pl.read_parquet(fold_path / "batch_*.parquet").to_pandas()

print("Загрузка фолдов v5 для Optuna...")
train_dfs = [load_fold(FEATURES_DIR / f"fold_{i:02d}") for i in range(4)]
train_df = pd.concat(train_dfs, ignore_index=True)
val_df = load_fold(FEATURES_DIR / "fold_04")

drop_cols = ["user_id", "anchor_date", "target"]
features = [c for c in val_df.columns if c not in drop_cols]

print(f"Train size: {len(train_df):,} | Val size: {len(val_df):,} | Features: {len(features)}")

X_train = train_df[features]
y_train_log = np.log1p(np.clip(train_df["target"].values, 0, None))

X_val = val_df[features]
y_val = val_df["target"].values
y_val_log = np.log1p(np.clip(y_val, 0, None))

train_pool = Pool(X_train, y_train_log)
val_pool = Pool(X_val, y_val_log)

Загрузка фолдов v5 для Optuna...
Train size: 1,000,000 | Val size: 250,000 | Features: 435


In [2]:
def objective(trial):
    params = {
        'iterations': trial.suggest_int('iterations', 2000, 4000, step=500),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.06, log=True),
        'depth': trial.suggest_int('depth', 6, 8),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1.0, 15.0, log=True),
        'random_strength': trial.suggest_float('random_strength', 0.1, 10.0, log=True),
        'bagging_temperature': trial.suggest_float('bagging_temperature', 0.0, 1.0),
        'loss_function': 'RMSE',
        'eval_metric': 'RMSE',
        'task_type': 'GPU',
        'devices': '0',
        'random_seed': 42,
        'od_type': 'Iter',
        'early_stopping_rounds': 150,
        'verbose': False,
    }
    
    model = CatBoostRegressor(**params)
    model.fit(train_pool, eval_set=val_pool)
    
    val_pred = np.expm1(np.clip(model.predict(X_val), 0, None))
    score = rmsle_score(y_val, val_pred)
    
    return score

In [3]:
N_OPTUNA_TRIALS = 25

def print_callback(study, trial):
    status = "PRUNED" if trial.state == optuna.trial.TrialState.PRUNED else f"RMSLE={trial.value:.5f}"
    print(f"  Trial {trial.number+1:2d}/{N_OPTUNA_TRIALS}: {status} "
          f"(best: {study.best_value:.5f}, trial #{study.best_trial.number+1})")

print(f"=== Запуск Optuna Tuning ({N_OPTUNA_TRIALS} итераций) ===")
study = optuna.create_study(direction="minimize", pruner=optuna.pruners.MedianPruner())
optuna.logging.set_verbosity(optuna.logging.WARNING)

study.optimize(objective, n_trials=N_OPTUNA_TRIALS, callbacks=[print_callback])

print(f"\nЛучший RMSLE на валидации: {study.best_value:.5f}")
print("Идеальные гиперпараметры:")
for k, v in study.best_params.items():
    print(f"  '{k}': {v},")

[I 2026-08-23 20:30:21,136] A new study created in memory with name: no-name-7eb26de0-322b-4c82-97d1-5fbf57af8d36


=== Запуск Optuna Tuning (25 итераций) ===
  Trial  1/25: RMSLE=1.65945 (best: 1.65945, trial #1)
  Trial  2/25: RMSLE=1.66573 (best: 1.65945, trial #1)
  Trial  3/25: RMSLE=1.66690 (best: 1.65945, trial #1)
  Trial  4/25: RMSLE=1.66410 (best: 1.65945, trial #1)
  Trial  5/25: RMSLE=1.66187 (best: 1.65945, trial #1)
  Trial  6/25: RMSLE=1.65634 (best: 1.65634, trial #6)
  Trial  7/25: RMSLE=1.67012 (best: 1.65634, trial #6)
  Trial  8/25: RMSLE=1.66797 (best: 1.65634, trial #6)
  Trial  9/25: RMSLE=1.66634 (best: 1.65634, trial #6)
  Trial 10/25: RMSLE=1.65899 (best: 1.65634, trial #6)
  Trial 11/25: RMSLE=1.66689 (best: 1.65634, trial #6)
  Trial 12/25: RMSLE=1.65938 (best: 1.65634, trial #6)
  Trial 13/25: RMSLE=1.65959 (best: 1.65634, trial #6)
  Trial 14/25: RMSLE=1.66198 (best: 1.65634, trial #6)
  Trial 15/25: RMSLE=1.65623 (best: 1.65623, trial #15)
  Trial 16/25: RMSLE=1.66308 (best: 1.65623, trial #15)
  Trial 17/25: RMSLE=1.65693 (best: 1.65623, trial #15)
  Trial 18/25: RMSL

# Генерация Event2Vec Эмбеддингов

In [1]:
import polars as pl
import pandas as pd
import numpy as np
from pathlib import Path
from gensim.models import Word2Vec
import time

DATA_DIR = Path("../data/raw")
PROCESSED = Path("../data/processed")

print("1. Загружаем сырые логи...")
t0 = time.time()
data = pl.read_parquet(DATA_DIR / 'train.parquet', columns=['user_id', 'event_date', 'searches', 'cat', 'to_cart', 'to_ord'])

print("2. Токенизация событий (Event Tokenization)...")

exprs = []
exprs.append(pl.when(pl.col("searches") > 0).then(pl.lit("search_")).otherwise(pl.lit("")))
exprs.append(pl.when(pl.col("cat") > 0).then(pl.lit("cat_")).otherwise(pl.lit("")))
exprs.append(pl.when(pl.col("to_cart") > 0).then(pl.lit("cart_")).otherwise(pl.lit("")))
exprs.append(pl.when(pl.col("to_ord") > 0).then(pl.lit("ord_")).otherwise(pl.lit("")))

data = data.with_columns(
    pl.concat_str(exprs).str.strip_chars_end("_").alias("daily_action")
)
data_active = data.filter(pl.col("daily_action") != "")

data_active = data_active.sort(["user_id", "event_date"])
data_active = data_active.with_columns(
    pl.col("event_date").diff().dt.total_days().over("user_id").fill_null(0).alias("gap_days")
)

data_active = data_active.with_columns(
    pl.when(pl.col("gap_days") > 30).then(pl.lit("30p"))
      .when(pl.col("gap_days") > 14).then(pl.lit("14p"))
      .when(pl.col("gap_days") > 7).then(pl.lit("7p"))
      .otherwise(pl.col("gap_days").cast(pl.Utf8))
      .alias("gap_token")
)

data_active = data_active.with_columns(
    (pl.col("daily_action") + "_gap" + pl.col("gap_token")).alias("event_word")
)

sentences_df = data_active.group_by("user_id").agg(
    pl.col("event_word").alias("sentence")
)

print(f"Подготовка завершена за {time.time()-t0:.1f} сек.")

print("3. Обучаем Word2Vec на поведении пользователей...")
sentences = sentences_df["sentence"].to_list()
user_ids = sentences_df["user_id"].to_list()

VECTOR_SIZE = 16

w2v_model = Word2Vec(
    sentences=sentences, 
    vector_size=VECTOR_SIZE, 
    window=5,
    min_count=2,
    workers=4,
    epochs=10
)

print("4. Создаем пользовательские векторы...")
user_embeddings = []

for seq in sentences:
    vecs = [w2v_model.wv[word] for word in seq if word in w2v_model.wv]
    if len(vecs) > 0:
        user_embeddings.append(np.mean(vecs[-10:], axis=0))
    else:
        user_embeddings.append(np.zeros(VECTOR_SIZE))

emb_matrix = np.array(user_embeddings)

emb_cols = [f"emb_action_{i}" for i in range(VECTOR_SIZE)]
emb_df = pl.DataFrame(emb_matrix, schema=emb_cols).with_columns(
    pl.Series("user_id", user_ids)
)

emb_df.write_parquet(PROCESSED / "user_action_embeddings.parquet")
print(f"Успех! Сгенерировано {VECTOR_SIZE} векторных признаков для {len(emb_df)} пользователей.")

1. Загружаем сырые логи...
2. Токенизация событий (Event Tokenization)...
Подготовка завершена за 11.8 сек.
3. Обучаем Word2Vec на поведении пользователей...
4. Создаем пользовательские векторы...
Успех! Сгенерировано 16 векторных признаков для 249616 пользователей.
